# Human Action Classifier (Single Model)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import re
import csv
import matplotlib.pyplot as plt

from pathlib import Path
from collections import Counter
from sklearn.cluster import MiniBatchKMeans
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from joblib import dump
from extract import extract_descriptors, gaussian_blur_3d, gradients_3d, second_moment_matrix, harris_response, detect_interest_points

In [ ]:
# Use GPU (CUDA/MPS) if available
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

### 1. Data Loading

In [ ]:
PREFIX = "0315/"
DESCRIPTION = "With Video Transformation (Horizontal Flip) and Empty Scene"
K = 800
ACTIONS = ["boxing", "handclapping", "handwaving", "jogging", "running", "walking", "empty"]
action_to_id = {a: i for i, a in enumerate(ACTIONS)}

VIDEO_ID_PATTERN = re.compile(r"person(?P<person>\d+)_(?P<action>[a-z]+)_d(?P<scene>\d+)")

def index_data(csv_file="data/info.csv", clips_root="data/transform"):
    samples = []

    with open(csv_file, newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            video_id = row["video_id"]
            target_video = row["target_video"]
            action = row["label"]
            isFlip = row["IsFlip"]

            match = VIDEO_ID_PATTERN.match(video_id)
            if match is None or action not in action_to_id:
                continue    # Skip video whose action isn't in our ACTIONS list

            # if isFlip == "T":
            #     continue  # Skip tranformed video

            clip_path = Path(clips_root) / target_video
            if not clip_path.exists():
                continue    # Skip missing clips (safe for partial generation)

            samples.append({
                "path": clip_path,
                "label": action_to_id[action],
                "action": action,
                "person": int(match.group("person")),
                "scene": int(match.group("scene")),
                "segment_id": int(row["segment_id"]),
                "video_id": video_id,
                "start_frame": int(row["start_frame"]),
                "end_frame": int(row["end_frame"]),
            })

    return samples

In [ ]:
# Import all data
samples = index_data()
print("Total:", Counter(s["action"] for s in samples))

# Train/validation/test split (70-30)
train_samples = [s for s in samples if s["person"] <= 18]
test_samples = [s for s in samples if s["person"] > 18]
print("Train:", len(train_samples), "Test:", len(test_samples))

### 2. Data Processing

#### 2.1. Descriptors Extraction

In [ ]:
def load_video(path, resize=(160, 120)):
    cap = cv2.VideoCapture(str(path))
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame = cv2.resize(frame, resize)
        frame = frame.astype(np.float32) / 255.0
        frames.append(frame)

    cap.release()
    video = np.stack(frames)   # (T, H, W)
    return torch.from_numpy(video)

def load_descriptors(filename, device="cpu"):
    # Load cached descriptors
    data = torch.load(filename, weights_only=False)
    descriptors = [d.to(device) for d in data["descriptors"]]
    print(f"Loaded descriptors from {filename}")
    return descriptors, data["paths"], data["labels"]

def save_descriptors(descriptors, samples, filename):
    # Save extracted descriptors
    data = {
        "paths": [s["path"] for s in samples],
        "labels": [s["label"] for s in samples],
        "descriptors": [d.cpu() for d in descriptors],
    }
    torch.save(data, filename)
    print(f"Saved descriptors to {filename}")


def extract_all_descriptors(samples, cache_file=None):
    # If cache_file exists, load instead
    if cache_file is not None and Path(cache_file).exists():
        descriptors, _, _ = load_descriptors(cache_file)
        return descriptors

    # Else, extract descriptors for all samples
    all_desc = []

    for s in samples:
        video = load_video(s["path"])
        video = video.to(device)
        desc = extract_descriptors(video)

        # store on CPU to save GPU memory
        all_desc.append(desc.cpu())

    # Save descriptors for later usage
    if cache_file is not None:
        save_descriptors(all_desc, samples, cache_file)

    return all_desc

def build_vocabulary(descriptors, K=400, max_samples=100_000):
    # Build vocabulary from pre-extracted training descriptors
    X = torch.cat(descriptors).cpu().numpy()

    if len(X) > max_samples:
        idx = np.random.choice(len(X), max_samples, replace=False)
        X = X[idx]

    kmeans = MiniBatchKMeans(n_clusters=K, batch_size=4096, random_state=0)
    kmeans.fit(X)
    return kmeans

In [ ]:
print("Extracting descriptors for all datasets...")
train_descriptors = extract_all_descriptors(train_samples, cache_file=f"artifact/{PREFIX}train_descriptors.pt")
test_descriptors = extract_all_descriptors(test_samples, cache_file=f"artifact/{PREFIX}test_descriptors.pt")

print("Learning vocabulary...")
kmeans = build_vocabulary(train_descriptors, K)
dump(kmeans, f"artifact/{PREFIX}kmeans_K{K}.joblib")

#### 2.2. Histogram Generation

In [ ]:
def encode_descriptors(desc, kmeans, K):
    # Encode pre-extracted descriptors as histogram
    if desc.numel() == 0:
        return torch.zeros(K)

    labels = kmeans.predict(desc.cpu().numpy())
    hist = np.bincount(labels, minlength=K).astype(np.float32)
    hist /= (hist.sum() + 1e-8)

    return torch.from_numpy(hist)

def build_dataset(samples, descriptors, kmeans, K):
    # Build dataset from pre-extracted descriptors
    X, y = [], []

    for i, s in enumerate(samples):
        # Encode descriptors
        hist = encode_descriptors(descriptors[i], kmeans, K)
        
        # Build dataset
        X.append(hist)
        y.append(s["label"])

    return torch.stack(X), torch.tensor(y)

In [ ]:
X_train, y_train = build_dataset(train_samples, train_descriptors, kmeans, K)
X_test, y_test = build_dataset(test_samples, test_descriptors, kmeans, K)

print("X_train", X_train.shape, "y_train", y_train.shape)
print("X_test", X_test.shape, "y_test", y_test.shape)

In [ ]:
save_dir = Path("artifact")
torch.save({
	"k": K,
	"actions": ACTIONS,
	"X_train": X_train, "y_train": y_train,
	"X_test": X_test, "y_test": y_test,
}, save_dir / f"{PREFIX}kth_features_K{K}.pt")

#### 2.3. Clustering Efficiency Measurement

In [ ]:
def compute_descriptor_kdr(descriptors, kmeans, max_samples=100000):
    # Compute normalized KDR in descriptor space (unsupervised)
    X = torch.cat(descriptors).cpu().numpy()
    Sw, Sb = 0, 0

    # Random sample descriptors to reduce running time
    if len(X) > max_samples:
        idx = np.random.choice(len(X), max_samples, replace=False)
        X = X[idx]

    # Get clusters
    labels = kmeans.predict(X)
    centers = kmeans.cluster_centers_
    global_mean = X.mean(axis=0)

    for k in range(K):
        cluster = X[labels == k]
        if len(cluster) == 0:
            continue

        # Calculate within cluster and between cluster variances
        mu_k = centers[k]
        n_k = len(cluster)
        Sw += ((cluster - mu_k) ** 2).sum()
        Sb += n_k * ((mu_k - global_mean) ** 2).sum()

    # Normalized KDR
    EPS = 1e-12
    kdr = (Sb / K) / (Sw / K + EPS)

    return kdr

def compute_feature_kdr(X, y):
    # Compute normalized KDR for histogram features (supervised)
    X, y = X.cpu().numpy(), y.cpu().numpy()
    Sw, Sb = 0, 0

    classes = np.unique(y)
    C = len(classes)
    global_mean = X.mean(axis=0)

    for c in classes:
        Xc = X[y == c]
        if len(Xc) == 0:
            continue

        # Calculate within cluster and between cluster variances
        mu_c = Xc.mean(axis=0)
        n_c = len(Xc)
        Sw += ((Xc - mu_c) ** 2).sum()
        Sb += n_c * ((mu_c - global_mean) ** 2).sum()

    # Normalized KDR
    EPS = 1e-12
    kdr = (Sb / C) / (Sw / C + EPS)

    return kdr

def compute_vocab_utilization(X):
    # Count number of used vocab
    active_counts = (X > 0).sum(dim=1).float()
    avg_active = active_counts.mean().item()
    std_active = active_counts.std().item()
    utilization_ratio = avg_active / K

    return avg_active, std_active, utilization_ratio

In [ ]:
descriptor_kdr = compute_descriptor_kdr(train_descriptors, kmeans)
print("Descriptor KDR:", descriptor_kdr)

feature_kdr = compute_feature_kdr(X_train, y_train)
print("Feature KDR:", feature_kdr)

_, _, utilization_ratio = compute_vocab_utilization(X_train)
print(f"Vocab Utilization Ratio: {utilization_ratio:.4f}")

#### 2.4. Data Visualization

##### 2.4.1. Across Dataset

In [ ]:
def compute_bow_statistics(X):
    # Compute BoW statistics for all data
    X = X.cpu().numpy()

    mean_per_word = X.mean(axis=0)
    std_per_word = X.std(axis=0)

    return mean_per_word, std_per_word

def visualize_statistics(mean_per_word, std_per_word, K):
    # Plot Bag-of-Words statistics for all data
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.hist(mean_per_word, bins=50)
    plt.title(f"Bag-of-Words Mean Distribution (K={K})")
    plt.xlabel("Mean activation")
    plt.ylabel("Count")
    plt.subplot(1,2,2)
    plt.hist(std_per_word, bins=50)
    plt.title(f"Bag-of-Words Standard Deviation Distribution (K={K})")
    plt.xlabel("Stdv activation")
    plt.tight_layout()
    plt.savefig(f"artifact/{PREFIX}stats_K{K}.png", dpi=150)
    plt.show()

In [ ]:
mean_w, std_w = compute_bow_statistics(X_train)
visualize_statistics(mean_w, std_w, K)

##### 2.4.2. Descriptor Frequency For Specific Frame

In [ ]:
def encode_video(video, kmeans, K):
    desc = extract_descriptors(video)

    if desc.numel() == 0:
        return torch.zeros(K)

    labels = kmeans.predict(desc.cpu().numpy())
    hist = np.bincount(labels, minlength=K).astype(np.float32)
    hist /= (hist.sum() + 1e-8)

    return torch.from_numpy(hist)

def visualize_histogram(hist, sample):
    # Visualize frequency histogram
    plt.figure(figsize=(12, 4))
    plt.bar(range(K), hist.numpy())
    plt.title(f"Action: {sample['action']} | Person: {sample['person']} | Scene: {sample['scene']}")
    plt.xlabel("Visual Word Index")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(f"artifact/{PREFIX}histogram_action{sample['action']}_person{sample['person']}_scene{sample['scene']}.png", dpi=150)
    plt.show()


In [ ]:
# Get 1 id per sequence (the first)
sequenceIds = []
for actions in ACTIONS:
    seqId = next(i for i, kv in enumerate(train_samples) if kv['action'] == actions)
    sequenceIds.append(seqId)

# Visualize histogram
for i in sequenceIds:
	sample = train_samples[i]
	video = load_video(sample["path"])
	video = video.to(device)
	hist = encode_video(video, kmeans, K)
	visualize_histogram(hist, sample)

#### 2.4.3. Descriptors on Specific Frame

In [ ]:
def visualize_descriptors(video, sample, sigma=1.5, tau=1.0, num_frames=2):
    # Extract features to get interest points
    L = gaussian_blur_3d(video, sigma, tau)
    Lx, Ly, Lt = gradients_3d(L)
    J = second_moment_matrix(Lx, Ly, Lt, 2*sigma, 2*tau)
    H = harris_response(J)
    points = detect_interest_points(H)
    print(f"Total descriptors detected: {len(points)}")
    if len(points) == 0:
        print("No descriptors detected in this video")
        return
    
    # Visualize descriptors on selected frames
    fig, axes = plt.subplots(1, min(num_frames, 2), figsize=(14, 5))
    if num_frames == 1:
        axes = [axes]
    
    # Select frames to visualize (spread across video)
    frame_indices = np.linspace(4, video.shape[0]-10, num_frames, dtype=int)
    
    for ax_idx, frame_idx in enumerate(frame_indices):
        frame = video[frame_idx].numpy()
        frame_display = (frame * 255).astype(np.uint8)
        
        # Get descriptors on this frame
        frame_points = points[points[:, 0] == frame_idx]
        
        ax = axes[ax_idx] if num_frames > 1 else axes[0]
        ax.imshow(frame_display, cmap='gray')
        
        # Draw circles at interest points
        if len(frame_points) > 0:
            for _, y, x in frame_points:
                circle = plt.Circle((x, y), 3, color='red', fill=False, linewidth=2)
                ax.add_patch(circle)
        
        ax.set_title(f"Frame {frame_idx} ({len(frame_points)} descriptors)")
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(f"artifact/{PREFIX}descriptors_action{sample['action']}_person{sample['person']}_scene{sample['scene']}.png", dpi=150)
    plt.show()

In [ ]:
# Visualize sample descriptors for each sequence (2 frames per seq)
for i in sequenceIds:
    sample = train_samples[i]
    video = load_video(sample["path"])
    video = video.cpu()
    visualize_descriptors(video, sample, num_frames=2)

### 3. Model Training

In [ ]:
class IntScaler():
    def __init__(self, scale=10000):
        self.scale = scale

    def fit_transform(self, data):
        return np.round(data * self.scale)
    
    def transform(self, data):
        return np.round(data * self.scale)

def train_svm(X, y):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X.numpy())

    clf = LinearSVC(C=1.0, max_iter=5000)
    clf.fit(Xs, y.numpy())

    return clf, scaler

def train_gnb(X, y):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    clf = GaussianNB()
    clf.fit(Xs, y.numpy())
    
    return clf, scaler

def train_mnb(X, y):
    scaler = IntScaler()
    Xs = scaler.fit_transform(X.numpy())

    clf = MultinomialNB()
    clf.fit(Xs, y.numpy())
    
    return clf, scaler

def train_mlp(X, y):
    mlp = MLPClassifier(hidden_layer_sizes=(512, 256), max_iter=1000)
    mlp.fit(X, y.numpy())
    return mlp, None

In [ ]:
print("Training classifier...")

clf_gnb, scaler_gnb = train_gnb(X_train, y_train)
clf_mnb, scaler_mnb = train_mnb(X_train, y_train)
clf_svm, scaler_svm = train_svm(X_train, y_train)
clf_mlp, scaler_mlp = train_mlp(X_train, y_train)

### 4. Model Evaluation

In [ ]:
def evaluate(clf, scaler, X, y):
    Xs = X if scaler is None else scaler.transform(X.numpy()) 
    y_pred = clf.predict(Xs)

    acc = balanced_accuracy_score(y.numpy(), y_pred)
    cm = confusion_matrix(y.numpy(), y_pred)

    return acc, cm

def visualize_cm(cm, model_name):
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=ACTIONS)
    disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
    plt.title(f"{model_name} Confusion Matrix")
    plt.tight_layout()
    plt.show()

def save_results(K, actions, test_accs, conf_mats, filename="results.txt", description=""):
    with open(filename, 'w') as f:
        f.write("="*60 + "\n")
        f.write("MODEL PERFORMANCE SUMMARY\n")
        f.write("="*60 + "\n")
        f.write(f"Description: {description}\n")
        f.write(f"Actions: {', '.join(actions)}\n")
        f.write(f"K: {K}\n")
        f.write(f"Descriptor KDR: {descriptor_kdr:.4f}\n")
        f.write(f"Feature KDR: {feature_kdr:.4f}\n")
        f.write(f"Vocab Utilization Ratio: {utilization_ratio:.4f}\n")
        f.write("-"*60 + "\n\n")

        f.write("TEST ACCURACY\n")
        for model_name, acc in test_accs.items():
            f.write(f"  {model_name:20s}: {acc*100:6.2f}%\n")
        f.write("\n" + "-"*60 + "\n")

        f.write("CONFUSION MATRICES\n")
        for model_name, cm in conf_mats.items():
            if torch.is_tensor(cm):
                cm = cm.cpu().numpy()
            f.write(f"{model_name}\n")
            # header row
            f.write(" " * 15)
            for a in actions:
                f.write(f"{a[:10]:>10}")
            f.write("\n")
            # matrix rows
            for i, row in enumerate(cm):
                f.write(f"{actions[i][:12]:>12}  ")
                for val in row:
                    f.write(f"{int(val):10d}")
                f.write("\n")
            f.write("\n")
    
    print(f"Results saved to {filename}")

In [ ]:
print("---- Test ----")

test_acc_gnb, cm_gnb = evaluate(clf_gnb, scaler_gnb, X_test, y_test)
print(f"Gaussian Naive Bayes test accuracy: {test_acc_gnb*100:.2f}%")
test_acc_mnb, cm_mnb = evaluate(clf_mnb, scaler_mnb, X_test, y_test)
print(f"Multinomial Naive Bayes test accuracy: {test_acc_mnb*100:.2f}%")
test_acc_svm, cm_svm = evaluate(clf_svm, scaler_svm, X_test, y_test)
print(f"Linear SVM test accuracy: {test_acc_svm*100:.2f}%")
test_acc_mlp, cm_mlp = evaluate(clf_mlp, scaler_mlp, X_test, y_test)
print(f"Multilayer Perceptron test accuracy: {test_acc_mlp*100:.2f}%")

# Save all results
test_accs = { 'Gaussian NB': test_acc_gnb, 'Multinomial NB': test_acc_mnb, 'Linear SVM': test_acc_svm, 'Multilayer Perceptron': test_acc_mlp}
test_cms = { 'Gaussian NB': cm_gnb, 'Multinomial NB': cm_mnb, 'Linear SVM': cm_svm, 'Multilayer Perceptron': cm_mlp}
save_results(K, ACTIONS, test_accs, test_cms, f"artifact/{PREFIX}results_K{K}.txt", description=DESCRIPTION)

# Visualize CM
for name in test_cms:
    visualize_cm(test_cms[name], name)